In [1]:
import os, rasterio, sys, rioxarray, pyflwdir, shutil, gc
import zipfile, dotenv, cdsapi, calendar
sys.path.append('backend/app/')
from rasterio.io import MemoryFile
from rasterio.features import rasterize, shapes
from rasterio.enums import Resampling
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions, functions
from shapely.geometry import Point, shape
from shapely import force_2d
from pyflwdir import dem
from tqdm import tqdm
from pathlib import Path
from whitebox.whitebox_tools import WhiteboxTools
from netCDF4 import Dataset, date2num
dotenv.load_dotenv()
wtb = WhiteboxTools()
wtb.set_verbose_mode(False)
np.random.seed(42)
from dateutil.relativedelta import relativedelta
from shapely.geometry import Point

In [3]:
sample_folder = r"backend\src\flow_samples"
test_folder = "test"
if not os.path.exists(test_folder): os.makedirs(test_folder)
catchment_path = os.path.join(sample_folder, 'catchment.geojson')
terrain_path = os.path.join(sample_folder, 'dtm10.tif')
lake_path = os.path.join(sample_folder, 'BrusdalLake.geojson')
river_path = os.path.join(sample_folder, 'river.geojson')
catchment = gpd.read_file(catchment_path)
# initial_param = [0.25,0.3,50,0,0,500,0,0,100]
raw_dir = os.path.normpath(os.path.join(f'{test_folder}/raw'))
raw_path = os.path.normpath(os.path.join(raw_dir, "dtm_raw.tif"))
lake = gpd.read_file(lake_path)
NODATA_FLWDR, NODATA_BASIN = 255, 0
terrain = rioxarray.open_rasterio(terrain_path).squeeze()
catchment_UTM = catchment.to_crs(terrain.rio.crs)

## Create a raw terrain that is clipped to catchment

In [4]:
# Clip terrain to catchment
if not os.path.exists(raw_dir): os.makedirs(raw_dir)
buffer = 10*terrain.rio.resolution()[0] if not terrain.rio.crs.is_geographic else 0.001
catchmentUTM_buffer = catchment_UTM.buffer(buffer)
minx, miny, maxx, maxy = catchmentUTM_buffer.total_bounds
terrain_clipped = terrain.rio.clip_box(minx, miny, maxx, maxy)
terrain_values, nodata = terrain_clipped.values, -9999.0
height, width = terrain_clipped.rio.height, terrain_clipped.rio.width
crs, transform = terrain_clipped.rio.crs, terrain_clipped.rio.transform()
lake_reproj = lake.to_crs(terrain_clipped.rio.crs)
lake_array = rasterize(
    shapes=[geom for geom in lake_reproj.geometry], dtype=np.float32,
    out_shape=(height, width), transform=transform, fill=nodata
)
mask_lake = (lake_array != nodata)
profile = {
    "driver": "GTiff", "count": 1, "dtype": np.float32, "crs": crs,
    "height": height, "width": width, "transform": transform
}
flow_functions.write_geotif(terrain_values, profile, raw_path, nodata)

## Create template hydro data

In [5]:
# Prepare template raster dataset
hydro_dir = os.path.normpath(f'{test_folder}/hydro')
if not os.path.exists(hydro_dir): os.makedirs(hydro_dir)
with rasterio.open(raw_path) as src:
    dem_array = src.read(1)
    crs, nodata = src.crs, src.nodata
    transform, profile = src.transform, src.profile
    height, width = src.height, src.width
# Fill depressions
filled_array, flwdir_array = dem.fill_depressions(elevtn=dem_array, max_depth=-1)
flw = pyflwdir.from_dem(filled_array, transform=transform, latlon=crs.is_geographic)
filled_array[mask_lake] = dem_array[mask_lake] # Replace lake elevation
# Create basins
basins_array = flw.basins()
unique, counts = np.unique(basins_array, return_counts=True)
largest_basin_id = unique[np.argmax(counts)]
basins_mask = (basins_array == largest_basin_id)
basins_array[basins_mask], basins_array[~basins_mask] = 1, NODATA_BASIN
elevtn_array = filled_array.copy()
# Create slope
dx, dy = transform.a, abs(transform.e)
# Gradient elevation
gradient_array = filled_array.copy()
gy, gx = np.gradient(gradient_array, dy, dx)
slope_array = np.sqrt(gx**2 + gy**2)
# Create stream order
uparea_array = flw.upstream_area(unit='km2')
# Create stream mask and stream order
stream_mask = (uparea_array > 10)
strord_array = flw.stream_order(type='strahler', mask=stream_mask)
# Create upstream grid
upstream_array = flw.upstream_area(unit='cell')
# Create river width
river = gpd.read_file(river_path).to_crs(crs)
river = river.rename(columns={'Width':"rivwth", 'Depth': 'rivdph'})
shape = ((geom, value) for geom, value in zip(river.geometry, river["rivwth"]))
rivwth_array = rasterize(
    shapes=shape, out_shape=(profile["height"], profile["width"]),
    transform=transform, fill=nodata, dtype=np.float32
)
files_float = {
    'elevtn.tif': [elevtn_array, nodata, np.float32],
    'flwdir.tif': [flwdir_array, NODATA_FLWDR, np.uint8],
    'lndslp.tif': [slope_array, nodata, np.float32],
    'basins.tif': [basins_array, NODATA_BASIN, np.int32], 
    'uparea.tif': [uparea_array, nodata, np.float32],
    'strord.tif': [strord_array, NODATA_BASIN, np.int16],
    'upgrid.tif': [upstream_array, NODATA_BASIN, np.int32],
    'rivwth.tif': [rivwth_array, nodata, np.float32]
}
profile_writer = profile.copy()
for file, array in files_float.items():
    profile_writer.update({'dtype': array[2]})
    flow_functions.write_geotif(array[0], profile_writer, os.path.join(hydro_dir, file), array[1])

## Process river data

In [6]:
# Create river path
river_dir = os.path.normpath(os.path.join(test_folder, 'river'))
if not os.path.exists(river_dir): os.makedirs(river_dir)
river_gpkg = os.path.normpath(os.path.join(river_dir, 'river.gpkg'))

In [7]:
# Create river from DEM if it doesn't exist
if not os.path.exists(river_path):
    threshold = 0.1 # threshold in km2
    with rasterio.open(raw_path) as src:
        dem_array = src.read(1).astype(np.float32)
        transform, crs, nodata = src.transform, src.crs, src.nodata
    # mask_nan = np.isnan(dem_array) | (dem_array == nodata)
    filled_array, _ = dem.fill_depressions(elevtn=dem_array, max_depth=-1)
    # flwdir_array = np.where(mask_nan, nodata, flwdir_array)
    flw = pyflwdir.from_dem(filled_array, transform=transform, latlon=crs.is_geographic)
    uparea = flw.upstream_area(unit="km2")
    river_mask = uparea > threshold
    features = flw.streams(river_mask)
    gdf = gpd.GeoDataFrame.from_features(features, crs=crs)
    # Clipp river to lake
    clipped_river = gdf.overlay(lake, how='difference')
    clipped_river['lenght'] = clipped_river['geometry'].length
    clipped_river.reset_index(drop=True, inplace=True)
    clipped_river = clipped_river[['geometry']]
    river = clipped_river.reindex(columns=['rivwth', 'rivdph', 'geometry'])
    # Create a random value for each river
    cols = {'rivwth': (0.05, 2), 'rivdph': (1, 5)}
    river_cols = river.columns.drop('geometry', errors='ignore')
    for col in river_cols:
        river[col] = pd.to_numeric(river[col], errors='coerce')
    for col, (low, high) in cols.items():
        river_mask = river[col].isna() | (river[col] == 'None')
        river.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
    river = river[river.is_valid].reset_index(drop=True)
    # Process river
    river["geometry"] = river.geometry.apply(lambda g: force_2d(g))
else:
    river = gpd.read_file(river_path)
    river = river.rename(columns={'Width':"rivwth", 'Depth': 'rivdph'})
    if not 'rivwth' in river.columns or not 'rivdph' in river.columns:
        raise Exception("river.gpkg must have rivwth and rivdph columns")
    river = river[['rivwth', 'rivdph', 'geometry']]
# Write file
river.to_file(river_gpkg, driver='GPKG')
manning_path = r"backend\src\flow_samples\river_manning_mapping.csv"
df = pd.read_csv(manning_path)
df.to_csv(os.path.join(river_dir, 'river_manning.csv'), index=False)

## Prepare forcing data from the customized area

In [ ]:
# Use one weather station to create the forcing data from CSV file
weather_path = os.path.join(sample_folder, 'weather_2025.csv')
start_time, end_time = "2025-01-01 00:00:00", "2025-02-01 00:00:00"
weather = pd.read_csv(weather_path)
weather['datetime'] = pd.to_datetime(weather['datetime'], format="%Y-%m-%d %H:%M:%S", utc=True)
start, end = pd.Timestamp(start_time, tz='UTC'), pd.Timestamp(end_time, tz='UTC')
weather_new = weather[weather['datetime'].between(start, end)].sort_values(by='datetime', ignore_index=True)
# # Resample
# weather_new = weather_new.resample('1H').interpolate(method='time')
# Create forcing nc file
time, time_step = weather_new['datetime'].dt.tz_localize(None).to_numpy(), 'hours'
forcing_dir = os.path.join(test_folder, 'forcing')
os.makedirs(forcing_dir, exist_ok=True)
out_path, datasets = os.path.join(forcing_dir, "weather_forcing.nc"), {}
with rasterio.open(raw_path) as src:
    dem_array, crs = src.read(1), src.crs
    transform, nodata = src.transform, src.nodata
mask_nan = np.isnan(dem_array) | (dem_array == nodata)
ny, nx = dem_array.shape[0], dem_array.shape[1]
chunksizes = (1, min(256, ny), min(256, nx))
forcing = {
    'precip': ['precip_mm', 'mm'], 'temp': ['temp_C', 'degC'],
    'kin': ['shortwave_Wm2', 'W/m^2'], 'kout': ['longwave_Wm2', 'W/m^2'],
    'wind': ['wind_mps', 'm/s'], 'press_msl': ['pressure', 'Pa']
}
for var, (col, unit) in forcing.items():
    data = weather_new[col].values.astype(np.float32)
    data_3d = flow_functions.create_forcing(data, ny, nx, mask_nan, True)
    datasets[var] = (('time', 'y', 'x'), data_3d, {'units': unit})
x_coords = transform.c + (np.arange(nx) + 0.5) * transform.a
y_coords = transform.f + (np.arange(ny) + 0.5) * transform.e
ds_final = xr.Dataset(
    data_vars=datasets, coords={"time": time, "y": y_coords, "x": x_coords}
)
ds_final = ds_final.rio.set_spatial_dims(x_dim="x", y_dim="y")
ds_final = ds_final.rio.write_crs(crs)
ds_final["time"].encoding = {
    "units": f"{time_step} since 1900-01-01 00:00:00",
    "calendar": "proleptic_gregorian", "dtype": "int64",
    "chunksizes": (len(time),), "zlib": True, 
    "complevel": 4, "shuffle": True,
}
encoding = {
    var: {"zlib": True, "complevel": 4, "shuffle": True, 
        "chunksizes": chunksizes, "_FillValue": -9999.0, "dtype": "float32"
    } for var in ds_final.data_vars
}
ds_final.to_netcdf(out_path, engine='netcdf4', encoding=encoding)

In [ ]:
# Prepare forcing data from the global model ARE5
# Source: https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download
CDS_url, CDS_key = os.getenv('CDS_URL'), os.getenv('CDS_API_KEY')
config_path = Path.home() / '.cdsapirc'
if not config_path.exists():
    print("Creating .cdsapirc ...")
    config_path.write_text(f"url: {CDS_url}\nkey: {CDS_key}\n", encoding='utf-8')
    print("Created at:", config_path)
catchment_WGS84 = catchment.to_crs('EPSG:4326')
# Setup variables
variables = {
    'total_precipitation': 'tp', # Precipitation
    '2m_temperature': 't2m', # Temperature
    '10m_u_component_of_wind': 'u10', '10m_v_component_of_wind': 'v10', # Wind
    'surface_pressure': 'sp',  # Pressure
    'surface_solar_radiation_downwards': 'ssrd', # Shortwave radiation
    'surface_thermal_radiation_downwards': 'strd', # Longwave radiation
}
forcing = {
    'precip': ['tp', 'mm'], 'temp': ['t2m', 'degC'],
    'kin': ['ssrd', 'W/m^2'], 'kout': ['strd', 'W/m^2'],
    'wind': ['', 'm/s'], 'press_msl': ['sp', 'Pa']
}
dataset = 'reanalysis-era5-single-levels'
start_time, end_time = '2025-01-01 00:00:00', '2025-02-01 00:00:00'
start, end = pd.Timestamp(start_time, tz='UTC'), pd.Timestamp(end_time, tz='UTC')
minx, miny, maxx, maxy = catchment_WGS84.total_bounds
area = [62.59501493381452, 6.237589637104367, 62.3482573134509, 6.6865812572157655]
forcing_dir = os.path.join(test_folder, 'forcing')
os.makedirs(forcing_dir, exist_ok=True)
download_dir = os.path.join(test_folder, 'forcing/download')
if not os.path.exists(download_dir): os.makedirs(download_dir)
forcing_path = os.path.join(forcing_dir, "weather_forcing.nc")
# Downloading data
with rasterio.open(raw_path) as src:
    dem_array, crs = src.read(1), src.crs
    transform, nodata = src.transform, src.nodata
mask_nan = np.isnan(dem_array) | (dem_array == nodata)
mask_nan = mask_nan.astype(bool)
ny, nx = dem_array.shape[0], dem_array.shape[1]
time_index, time_step = 0, 'hours'
x_coords = transform.c + (np.arange(nx) + 0.5) * transform.a
y_coords = transform.f + (np.arange(ny) + 0.5) * transform.e
nc = Dataset(forcing_path, "w", format="NETCDF4")
nc.createDimension("time", None)
nc.createDimension("y", ny)
nc.createDimension("x", nx)
time_var = nc.createVariable("time", "f8", ("time",))
time_var.units = f"{time_step} since 1900-01-01 00:00:00"
time_var.calendar = "proleptic_gregorian"
y_var = nc.createVariable("y", "f4", ("y",))
x_var = nc.createVariable("x", "f4", ("x",))
y_var[:], x_var[:], nc_vars = y_coords, x_coords, {}
crs_var = nc.createVariable("crs", "i4")
if crs.is_geographic: crs_var.grid_mapping_name = "latitude_longitude"
else: crs_var.grid_mapping_name = "transverse_mercator"
crs_var.crs_wkt = crs.to_wkt()
try: crs_var.epsg_code = crs.to_epsg()
except Exception: pass
for var, (_, unit) in forcing.items():
    nc_vars[var] = nc.createVariable(
        var, "f4", ("time", "y", "x"), zlib=True, complevel=7, fill_value=-9999.0,
        shuffle=True, chunksizes=(24, 128, 128), least_significant_digit=2
    )
    nc_vars[var].units, nc_vars[var].grid_mapping = unit, "crs"
client = cdsapi.Client(quiet=False, debug=False)
current = start.replace(day=1)
while current <= end:
    files, bad_files, month_files = [], [], []
    year, month = current.year, current.month
    last_day = calendar.monthrange(year, month)[1]
    month_start = datetime(year, month, 1)
    month_end = datetime(year, month, last_day, 23)
    # Clip by requested range
    actual_start = max(start, month_start)
    actual_end = min(end, month_end)
    # Days to download
    days = [f"{d:02d}" for d in range(actual_start.day, actual_end.day + 1)]
    for key, var in variables.items():
        request = {
            'product_type': 'reanalysis', 'variable': [key],
            'year': [str(year)], 'month': [f"{month:02d}"], 'day': days,
            'time': [f"{h:02d}:00" for h in range(24)], 'area': area,
            'data_format': 'netcdf', 'download_format': 'unarchived'
        }
        out_file = f"{year}_{month:02d}_{var}.nc"
        out_path = os.path.join(download_dir, out_file)
        client.retrieve(dataset, request, out_path)
        # Check valid files
        if flow_functions.is_valid_netcdf(out_path, var): files.append(out_path)
        else: bad_files.append(out_path)
        month_files.append(out_path)
    print(f"Year: {year}, Month: {month}")
    print(f"Valid files: {len(files)}/{len(files)+len(bad_files)}")
    print(f"Invalid files: {len(bad_files)}/{len(files)+len(bad_files)}")
    if bad_files:
        print("Bad files:")
        for f in bad_files: print(" -", f)
    print("Processing monthly forcing ...")
    ref_file = next(f for f in month_files if f.endswith("_tp.nc"))
    with xr.open_dataset(ref_file) as ref_ds:
        timestamps = pd.to_datetime(ref_ds['valid_time'][:]).to_numpy()
        lat, lon = ref_ds['latitude'][:], ref_ds['longitude'][:]
    x_known, y_known, gdf = None, None, None
    single_value = lat.size * lon.size == 1
    if not single_value:
        lon2d, lat2d = np.meshgrid(lon, lat)
        gdf_known = gpd.GeoDataFrame(
            geometry=gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()), crs='EPSG:4326'
        ).to_crs(crs)
        x_known, y_known = gdf_known.geometry.x.values, gdf_known.geometry.y.values
        x, y = np.meshgrid(x_coords, y_coords)
        gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy(x.ravel(), y.ravel()), crs=crs)
        idx, weight = flow_functions.prepare_interpolator(gdf, x_known, y_known, geo_type="point")
    n_time, chunk_size, overlap = len(timestamps), 24, 2
    for var, (col, unit) in forcing.items():
        print(f"Processing {var}")
        if var == 'wind':
            u_file = os.path.join(download_dir,f"{year}_{month:02d}_u10.nc")
            v_file = os.path.join(download_dir,f"{year}_{month:02d}_v10.nc")
            ds_u, ds_v = Dataset(u_file), Dataset(v_file)
        else: ds = Dataset(os.path.join(download_dir, f"{year}_{month:02d}_{col}.nc"))
        for start in range(0, n_time, chunk_size):
            start_eff = max(0, start - overlap)
            end_eff = min(n_time, start + chunk_size + overlap)
            if var == "wind":
                u = ds_u["u10"][start_eff:end_eff].astype(np.float32)
                v = ds_v["v10"][start_eff:end_eff].astype(np.float32)
                data = np.hypot(u, v)
            else: data = ds[col][start_eff:end_eff].astype(np.float32)
            if var == "precip": data *= 1000
            elif var == "temp": data -= 273.15
            elif var in ("kin", "kout"): data /= 3600
            if np.isnan(data).any():
                da = xr.DataArray(data, dims=("valid_time", "latitude", "longitude"))
                da = da.interpolate_na(dim="valid_time", method="linear")
                da = da.ffill("valid_time").bfill("valid_time")
                da = da.fillna(0.0)
                data = da.values.astype(np.float32)
            # Interpolate
            data_3d = flow_functions.create_forcing(data, ny, nx, mask_nan, single_value, idx, weight)
            t0, t1 = start, min(start + chunk_size, n_time)
            chunk_start = t0 - start_eff
            chunk_end = chunk_start + (t1 - t0)
            chunk = data_3d[chunk_start:chunk_end]
            chunk = np.nan_to_num(chunk, nan=0.0)
            chunk = np.where(mask_nan[None, :, :], 0.0, chunk)
            nc_vars[var][t0:t1, :, :] = chunk
            del data, data_3d, chunk
            if var == "wind": del u, v
        if var == "wind":
            ds_u.close()
            ds_v.close()
        else: ds.close()
    for i, t in enumerate(timestamps):
        time_var[time_index + i] = date2num(pd.Timestamp(t).to_pydatetime(), time_var.units, time_var.calendar)
    time_index += n_time
    # Cleanup
    gc.collect()
    for f in month_files: functions.safe_remove(f)
    # Next month
    current += relativedelta(months=1)
nc.close()
if os.path.exists(download_dir): shutil.rmtree(download_dir)

## Process land cover

In [9]:
# Create land cover nc file
land_dir = os.path.normpath(os.path.join(test_folder, 'landcover'))
if not os.path.exists(land_dir): os.makedirs(land_dir)
terrain_da = rioxarray.open_rasterio(raw_path).squeeze()
with rasterio.open(raw_path) as src:
    nodata, profile, dem_array = src.nodata, src.profile, src.read(1)
profile_writer = profile.copy()
mask_nan = np.isnan(dem_array) | (dem_array == nodata)

In [10]:
# Land cover data from CORINE 2018
corine_path = r"backend\src\flow_samples\landcover\U2018_CLC2018_V2020_20u1.zip"
with zipfile.ZipFile(corine_path, "r") as zip_ref:
    name = os.path.basename(corine_path).replace(".zip", ".tif")
    with zip_ref.open(name) as f:
        with MemoryFile(f.read()) as memfile:
            corine = rioxarray.open_rasterio(memfile).squeeze()
            corine_match = corine.rio.reproject_match(terrain_da, resampling=Resampling.nearest)
land_corine = corine_match.values.astype(np.uint8)
land_corine[mask_nan] = NODATA_FLWDR
path = os.path.join(land_dir, 'corine.tif')
profile_writer.update(dtype=np.uint8)
flow_functions.write_geotif(land_corine, profile_writer, path, NODATA_FLWDR)
# Save look up table
csv_path = r"backend\src\flow_samples\landcover\corine_mapping.csv"
df, lai_name = pd.read_csv(csv_path), "corine"
mask_csv = df['corine'] != 999
df.loc[~mask_csv, 'corine'], df.loc[~mask_csv, 'landuse'] = 48, -999.0
df.loc[mask_csv, 'corine'] = np.arange(1, len(df.loc[mask_csv, 'corine']) + 1).astype(np.uint8)
df.to_csv(os.path.join(land_dir, 'corine.csv'), index=False)
# Compute LAI
lai_path = os.path.join(land_dir, "corine_lai.csv")
flow_functions.create_LAI(land_corine, terrain_da, lai_path, NODATA_FLWDR)

## Process soil data

In [11]:
# Initialize variables
soil_dir = os.path.join(f'{test_folder}/soil')
if not os.path.exists(soil_dir): os.makedirs(soil_dir)
catchment_WGS84 = catchmentUTM_buffer.to_crs('EPSG:4326')
min_lon, min_lat, max_lon, max_lat = catchment_WGS84.total_bounds
soil_types = {
    'bulk density': ['BLDFIE_M', 'bd', 1000, np.int16, -32768],
    'clay': ['CLYPPT_M', 'clyppt', 1, np.uint8, 255], 
    'sand': ['SNDPPT_M', 'sndppt', 1, np.uint8, 255],
    'silt': ['SLTPPT_M', 'sltppt', 1, np.uint8, 255],  
    'organic carbon': ['OCDENS_M', 'oc', 1, np.int16, -32768], 
    'pH': ['PHIHOX_M', 'ph', 10, np.uint8, 255]
}
depths = ['sl1', 'sl2', 'sl3', 'sl4', 'sl5', 'sl6', 'sl7']
# Get raster information
ref = rioxarray.open_rasterio(raw_path).squeeze()
with rasterio.open(raw_path) as src:
    profile, transform = src.profile, src.transform
    height, width, dem_array = src.height, src.width, src.read(1)
    nodata, crs = src.nodata, src.crs
# Download soil data 2017 from ISRIC: https://files.isric.org/soilgrids/former/2017-03-10/
base_url = "https://files.isric.org/soilgrids/former/2017-03-10/data/"
# Soil thickness
soil_thickness_url, NODATA_SOIL_THICKNESS = f"{base_url}BDRICM_M_250m_ll.tif", -99999
with rioxarray.open_rasterio(soil_thickness_url) as src:
    data_xr = src.rio.clip_box(minx=min_lon, miny=min_lat, maxx=max_lon, maxy=max_lat)
    data_xr = data_xr.rio.reproject_match(ref, resampling=Resampling.nearest)
    nodata = data_xr.rio.nodata
soil_thickness_array = data_xr[0].values
# Interpolate data
mask_valid = soil_thickness_array != nodata
soil_thickness_values = flow_functions.interpolate_extrapolate(soil_thickness_array, ~mask_valid, True)
soil_thickness_values = soil_thickness_values.astype(np.int32)
soil_thickness_values[mask_lake] = NODATA_SOIL_THICKNESS
soil_thickness_path = os.path.normpath(os.path.join(soil_dir, 'soilthickness.tif'))
profile_writer = profile.copy()
profile_writer.update(dtype=np.int32)
flow_functions.write_geotif(soil_thickness_values, profile_writer, soil_thickness_path, NODATA_SOIL_THICKNESS)
# Download soil data
for item, values in tqdm(soil_types.items(), total=len(soil_types), desc='Downloading 2017 soil data'):
    file, name, scale, dtype, nodata_soil, bulk_density = values[0], values[1], values[2], values[3], values[4], None
    for depth in depths:
        file_url = f'{base_url}{file}_{depth}_250m_ll.tif'
        with rioxarray.open_rasterio(file_url) as src:
            data_xr = src.rio.clip_box(minx=min_lon, miny=min_lat, maxx=max_lon, maxy=max_lat)
            data_xr = data_xr.rio.reproject_match(ref, resampling=Resampling.nearest)
            nodata = data_xr.rio.nodata
        soil_array = data_xr[0].values
        # Interpolate data
        mask_valid = soil_array != nodata
        soil_values = flow_functions.interpolate_extrapolate(soil_array, ~mask_valid, True)
        if name == 'BLDFIE_M': bulk_density = soil_values + 1e-6
        if name == 'OCDENS_M':
            # Convert Organic Carbon Density (kg/m³) to Organic Carbon Content (%)
            soil_values = soil_values / bulk_density * 100
        soil_values = soil_values / scale
        soil_values[mask_lake] = nodata_soil
        path = os.path.join(soil_dir, f'{name}_{depth}.tif')
        profile_writer.update(dtype=dtype)
        flow_functions.write_geotif(soil_values, profile_writer, path, nodata_soil)
